In [0]:
%pip install geopandas shapely

In [0]:

import pandas as pd
from shapely.geometry import box
from shapely import wkt
import numpy as np

In [0]:
import geopandas as gpd
import pandas as pd

from shapely.geometry import box
from shapely import wkt

# --------------------------------------------------
# ADLS paths
# --------------------------------------------------

SILVER_HAZARD_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/fire_hazard"
)

SILVER_ZONES_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/analysis_zones"
)

# --------------------------------------------------
# Study area
# --------------------------------------------------

STUDY_AREA_NAME = "Western Los Angeles County"

MIN_LONGITUDE = -118.95
MIN_LATITUDE = 33.95
MAX_LONGITUDE = -118.45
MAX_LATITUDE = 34.25

# --------------------------------------------------
# Coordinate systems
# --------------------------------------------------

WGS84_CRS = "EPSG:4326"
PROJECTED_CRS = "EPSG:3310"

# --------------------------------------------------
# Grid configuration
# --------------------------------------------------

GRID_SIZE_METERS = 2000
MIN_LAND_RATIO = 0.20

print("Configuration loaded.")

In [0]:
hazard_pdf = (
    spark.read
    .format("delta")
    .load(SILVER_HAZARD_PATH)
    .toPandas()
)

hazard_pdf["geometry"] = (
    hazard_pdf["geometry_wkt"]
    .apply(wkt.loads)
)

hazard_gdf = gpd.GeoDataFrame(
    hazard_pdf.drop(columns=["geometry_wkt"]),
    geometry="geometry",
    crs=WGS84_CRS
)

print("Hazard rows:", len(hazard_gdf))
print("CRS:", hazard_gdf.crs)

In [0]:
study_boundary = gpd.GeoDataFrame(
    {
        "study_area": [STUDY_AREA_NAME]
    },
    geometry=[
        box(
            MIN_LONGITUDE,
            MIN_LATITUDE,
            MAX_LONGITUDE,
            MAX_LATITUDE
        )
    ],
    crs=WGS84_CRS
)

print(study_boundary)

In [0]:
boundary_projected = (
    study_boundary
    .to_crs(PROJECTED_CRS)
)

hazard_projected = (
    hazard_gdf
    .to_crs(PROJECTED_CRS)
)

minx, miny, maxx, maxy = (
    boundary_projected.total_bounds
)

print("Projected bounds:")
print(minx, miny, maxx, maxy)

In [0]:
grid_cells = []

x = minx

while x < maxx:

    y = miny

    while y < maxy:

        cell = box(
            x,
            y,
            x + GRID_SIZE_METERS,
            y + GRID_SIZE_METERS
        )

        grid_cells.append(cell)

        y += GRID_SIZE_METERS

    x += GRID_SIZE_METERS


grid = gpd.GeoDataFrame(
    geometry=grid_cells,
    crs=PROJECTED_CRS
)

print("Initial grid cells:", len(grid))

In [0]:
grid = gpd.clip(
    grid,
    boundary_projected
)

grid = grid[
    ~grid.geometry.is_empty
].copy()

grid = grid.reset_index(drop=True)

print(
    "Grid cells after boundary clip:",
    len(grid)
)

In [0]:
study_area_geometry = (
    hazard_projected
    .geometry
    .union_all()
)

print(
    "Hazard footprint created:",
    not study_area_geometry.is_empty
)

In [0]:
grid["cell_area"] = (
    grid.geometry.area
)

grid["study_area_overlap"] = (
    grid.geometry.apply(
        lambda geom:
        geom.intersection(
            study_area_geometry
        ).area
    )
)

grid["land_ratio"] = (
    grid["study_area_overlap"]
    /
    grid["cell_area"]
)

print(
    grid["land_ratio"]
    .describe()
)

In [0]:
zones = grid[
    grid["land_ratio"]
    >= MIN_LAND_RATIO
].copy()

zones = zones.reset_index(drop=True)

print(
    "Usable analysis zones:",
    len(zones)
)

print(
    "Land ratio range:",
    round(zones["land_ratio"].min(), 3),
    "to",
    round(zones["land_ratio"].max(), 3)
)

In [0]:
zones_final = gpd.clip(
    zones,
    study_area_geometry
)

zones_final = zones_final[
    ~zones_final.geometry.is_empty
].copy()

zones_final = (
    zones_final
    .reset_index(drop=True)
)

zones_final["zone_id"] = [
    f"ZONE_{i+1:03d}"
    for i in range(
        len(zones_final)
    )
]

print(
    "Final analysis zones:",
    len(zones_final)
)

In [0]:
zones_final = zones_final[
    [
        "zone_id",
        "geometry"
    ]
].copy()

In [0]:
print(
    "Rows:",
    len(zones_final)
)

print(
    "Unique zone IDs:",
    zones_final["zone_id"].nunique()
)

print(
    "Invalid geometries:",
    (~zones_final.geometry.is_valid).sum()
)

print(
    "Empty geometries:",
    zones_final.geometry.is_empty.sum()
)

print(
    "CRS:",
    zones_final.crs
)

assert len(zones_final) == 227
assert zones_final["zone_id"].nunique() == 227
assert (~zones_final.geometry.is_valid).sum() == 0
assert zones_final.geometry.is_empty.sum() == 0

print("✅ Analysis zone validation passed.")

In [0]:
zones_to_save = zones_final.copy()

zones_to_save["geometry_wkt"] = (
    zones_to_save.geometry.apply(
        lambda geom:
        geom.wkt
        if geom is not None
        else None
    )
)

zones_to_save = (
    zones_to_save[
        [
            "zone_id",
            "geometry_wkt"
        ]
    ]
)

print(
    "Rows prepared for Silver:",
    len(zones_to_save)
)

In [0]:
zones_spark_df = spark.createDataFrame(
    zones_to_save
)

(
    zones_spark_df.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_ZONES_PATH)
)

print(
    "Silver analysis zones written successfully."
)

print(
    "Path:",
    SILVER_ZONES_PATH
)

In [0]:
check_zones = (
    spark.read
    .format("delta")
    .load(SILVER_ZONES_PATH)
)

print(
    "Rows written:",
    check_zones.count()
)

print(
    "Unique zone IDs:",
    check_zones
    .select("zone_id")
    .distinct()
    .count()
)

display(
    check_zones.limit(5)
)